# Tool 살펴보기 — 에이전트에 어떤 도구를 붙일 수 있나

**Tool(도구)** 은 LLM 이 스스로 호출해 외부 기능을 쓰게 해주는 함수다. LLM 은 학습 시점 이후 정보·실시간 데이터·계산·DB 조회 등을 직접 못 하는데, 도구를 쥐여주면 "이 도구를 이런 인자로 불러주세요" 라는 **호출 의도(tool_calls)** 를 만들어낸다. 실제 실행은 우리가(또는 `ToolNode` 가) 한다.

이 노트북에서 다루는 것:
1. 도구는 무엇으로 구성되는가 (name / description / args)
2. 직접 도구 만들기 — `@tool`, `args_schema`, `StructuredTool`
3. 붙일 수 있는 도구 종류 카탈로그 (검색 / 위키·논문 / 리트리버 / 외부연동)
4. `ToolNode` 로 그래프 안에서 도구 실행하기

> 이 노트북의 핵심 예제(직접 만든 도구·ToolNode)는 **LLM/API 키 없이** 돌아간다.
> 외부 검색·위키 도구는 네트워크가 필요하고, LLM 에 실제로 붙여 쓰는 전체 흐름은 `projects/02_tool_calling_web_search.ipynb` 에서 다룬다.

## 0. 도구의 3요소

LLM 은 도구의 **이름(name)** · **설명(description)** · **입력 스키마(args)** 를 보고 "이 질문에 이 도구가 맞겠다" 를 판단한다. 그래서 도구를 만들 때 **함수명·docstring·타입힌트** 가 곧 LLM 에게 주는 설명서다.

| 요소 | 무엇 | 어디서 옴 |
|---|---|---|
| `name` | 도구 식별자 | 함수 이름 |
| `description` | 언제 쓰는 도구인지 | docstring |
| `args` | 입력 인자와 타입 | 함수 시그니처 / `args_schema` |

## 1. 직접 도구 만들기

### 1-1. `@tool` 데코레이터 (가장 간단)

일반 함수에 `@tool` 을 붙이면 도구가 된다. **docstring 은 필수** — LLM 이 이걸 보고 도구를 고른다.

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """두 정수 a, b 를 곱한다."""   # 이 docstring 이 곧 LLM 에게 주는 설명
    return a * b

# 도구의 3요소 확인
print("name       :", multiply.name)
print("description:", multiply.description)
print("args       :", multiply.args)

# 도구를 직접 실행해보기 (LLM 없이도 그냥 함수처럼 호출 가능)
print("invoke     :", multiply.invoke({"a": 6, "b": 7}))

### 1-2. 입력 스키마 명시 (`args_schema`)

인자에 설명·기본값·제약을 주고 싶으면 Pydantic 모델로 입력 스키마를 정의해 붙인다. [basics 복습] Pydantic `BaseModel` 의 `Field` 로 각 인자에 description 을 단다 — 이 설명도 LLM 이 읽는다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool

class SearchInput(BaseModel):
    query: str = Field(description="검색할 키워드")
    top_k: int = Field(default=3, description="가져올 결과 개수")

@tool(args_schema=SearchInput)
def local_search(query: str, top_k: int = 3) -> str:
    """로컬 더미 검색기. 실제 네트워크 호출 없이 형식만 보여준다."""
    return f"[{query!r} 에 대한 결과 {top_k}건]"

print("args  :", local_search.args)
print("invoke:", local_search.invoke({"query": "langgraph", "top_k": 2}))

### 1-3. `StructuredTool.from_function` (기존 함수를 도구로)

이미 있는 함수를 데코레이터 없이 도구로 감쌀 때 쓴다. 동기/비동기 함수 모두 가능.

In [ ]:
from langchain_core.tools import StructuredTool

def add(a: int, b: int) -> int:
    """두 정수를 더한다."""
    return a + b

add_tool = StructuredTool.from_function(add)
print(add_tool.name, "->", add_tool.invoke({"a": 10, "b": 5}))

## 2. Tool Calling 인터페이스 — `bind_tools` 가 하는 일

도구를 만들었다고 LLM 이 알아서 쓰는 게 아니다. **`llm.bind_tools([...])`** 로 도구 목록을 LLM 에 쥐여줘야, LLM 이 사용자 입력을 보고 "이 도구를 이런 인자로 부르면 되겠다" 를 판단할 수 있다.

핵심: **LLM 의 역할은 도구를 *실행*하는 게 아니라, 자연어에서 도구 호출에 필요한 `arguments` 를 *감지*하는 것**이다.

```
  자연어 입력                  [ LLM (도구가 bind 됨) ]            도구 호출 페이로드
  "What is 2 times 3"  ───▶                                ───▶  name: "multiply"
                                  🧠  자연어 → arguments 감지         arguments: {"a": 2, "b": 3}
  @tool                      ▲
  def multiply(a, b):  ──────┘  (StructuredTool 로 변환되어 bind)
      return a * b
```

- 입력 = 사용자의 **자연어** + bind 된 **도구 정의(StructuredTool)**
- 출력 = **도구 호출 페이로드** (`name` + `arguments`) — 아직 *실행 전*, "이렇게 부르면 된다"는 의도일 뿐
- 웹검색 도구라면: `"What is LangGraph?"` → `name: "tavily_search"`, `arguments: {"query": "LangGraph"}`

### `bind_tools` 호출해보기 (키 필요)

`bind_tools` 의 결과를 invoke 하면, 응답 AIMessage 의 **`.tool_calls`** 에 위 페이로드가 담긴다.
도구가 필요 없는 일반 질문이면 `.tool_calls` 는 비어 있고 그냥 답한다.

> `OPENAI_API_KEY` 가 있어야 실행된다. 없으면 셀이 안내만 출력한다.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o")
    llm_with_tools = llm.bind_tools([multiply, add_tool])   # 도구를 LLM 에 바인딩

    # 1) 도구가 필요한 질문 → tool_calls 에 호출 페이로드(name + args) 생성
    res = llm_with_tools.invoke("What is 2 times 3?")
    print("tool_calls:", res.tool_calls)        # [{'name':'multiply','args':{'a':2,'b':3'}, ...}]
    print("content   :", repr(res.content))      # 보통 비어 있음 (아직 도구 실행 전)

    # 2) 도구가 필요 없는 질문 → tool_calls 비어 있고 그냥 답함
    res2 = llm_with_tools.invoke("안녕? 한 단어로 인사해줘")
    print("\n일반 질문 tool_calls:", res2.tool_calls)
    print("일반 질문 content   :", res2.content)
else:
    print("OPENAI_API_KEY 가 없어 건너뜀 (.env 에 키를 넣으면 실행됨)")

## 3. 사용자 질의 → 최종 응답: 전체 흐름

도구를 가진 에이전트에서 질문 하나가 답이 되기까지 거치는 단계다. [basics 복습] 이 흐름은 `chatbot` 노드와 `tools` 노드가 조건부 엣지로 연결된 **루프** 로 구현된다.

```
  ① 사용자 질의 (자연어)
        │
        ▼
  ② chatbot 노드: bind 된 LLM 호출
        │
        ├─ 도구 필요 없음 ──────────────────────────▶ ⑥ 최종 응답 → END
        │
        └─ 도구 필요함 (AIMessage.tool_calls 생성)
              │  ③ 조건부 라우팅: tool_calls 있으면 tools 로
              ▼
        ④ tools 노드 (ToolNode): 도구를 실제 실행 → ToolMessage(결과)
              │  ⑤ tools → chatbot 으로 되돌아감 (결과를 들고)
              ▼
        ② chatbot 노드: 도구 결과를 보고 다시 판단
              └─ 더 부를 도구 없으면 ─────────────────▶ ⑥ 최종 응답 → END
```

| 단계 | 무슨 일 | 만들어지는 메시지 |
|---|---|---|
| ① 질의 | 사용자가 자연어로 질문 | `HumanMessage` |
| ② 판단 | LLM 이 도구 필요 여부 판단 (`bind_tools`) | `AIMessage` (+ `tool_calls`) |
| ③ 라우팅 | `tool_calls` 유무로 분기 (조건부 엣지) | — |
| ④ 실행 | `ToolNode` 가 도구 실행 | `ToolMessage` (결과) |
| ⑤ 되돌림 | 결과를 들고 LLM 으로 복귀 (`tools → chatbot`) | — |
| ⑥ 응답 | LLM 이 도구 결과를 녹여 최종 답변 | `AIMessage` (최종) |

이 루프 덕분에 **도구를 여러 번** 부를 수도 있다 (검색 → 결과 보고 추가 검색 → …). 더 부를 도구가 없으면 LLM 이 최종 답을 만들고 END 로 빠진다.

> 이 흐름을 실제 코드로 조립하는 전체 예제는 `projects/02_tool_calling_web_search.ipynb` 에 있다. 여기 `ToolNode` 절(아래)에서는 그중 ④ 실행 단계만 LLM 없이 떼어내 확인한다.

## 4. 붙일 수 있는 도구 카탈로그

도구는 크게 ① 직접 만든 커스텀 도구, ② 라이브러리가 제공하는 사전구축(prebuilt) 도구로 나뉜다. LangChain/LangGraph 생태계에는 수백 개의 prebuilt 도구가 있고, 역할별로 정리하면 다음과 같다.

### 검색 · 정보 조회

| 도구 | 역할 | import | 키/패키지 |
|---|---|---|---|
| `TavilySearch` | LLM 친화적 웹검색 (실시간 정보) | `langchain_tavily` | `TAVILY_API_KEY` |
| `DuckDuckGoSearchRun` | 키 없이 쓰는 웹검색 | `langchain_community.tools` | `duckduckgo-search` |
| `WikipediaQueryRun` | 위키백과 문서 조회 | `langchain_community.tools` | `wikipedia` |
| `ArxivQueryRun` | arXiv 논문 검색/요약 | `langchain_community.tools.arxiv.tool` | `arxiv` |
| `YouTubeSearchTool` | 유튜브 영상 검색 | `langchain_community.tools` | `youtube-search` |

### 코드 실행 · 시스템

사용자가 코드를 요청하면 **실제로 실행해서 결과를 검증**하거나, 셸 명령·HTTP 요청을 수행하는 도구.

| 도구 | 역할 | import | 비고 |
|---|---|---|---|
| `PythonREPLTool` | 파이썬 코드를 실제 실행하고 결과 반환 | `langchain_experimental.tools` | ⚠️ 임의 코드 실행 — 샌드박스 권장 |
| `ShellTool` | 셸 명령 실행 | `langchain_community.tools` | ⚠️ 위험, 격리 환경 권장 |
| `RequestsGetTool` 등 | HTTP 요청(GET/POST/…) | `langchain_community.tools.requests.tool` | `requests` |

### 외부 SaaS 연동 (Toolkit)

여러 도구가 묶인 **Toolkit** 형태로 제공된다. `toolkit.get_tools()` 로 도구 리스트를 얻어 LLM 에 bind 한다.

| Toolkit | 역할 | import | 키 |
|---|---|---|---|
| `GitHubToolkit` | 이슈/PR 생성·조회, 파일 커밋 등 GitHub 연동 | `langchain_community.agent_toolkits.github.toolkit` | GitHub App 토큰 |
| `SlackToolkit` | 메시지 전송·채널 조회 | `langchain_community.agent_toolkits` | Slack 토큰 |
| `GmailToolkit` | 메일 조회·작성·발송 | `langchain_community.agent_toolkits` | Google OAuth |
| `SQLDatabaseToolkit` | DB 스키마 조회·SQL 실행 | `langchain_community.agent_toolkits.sql.toolkit` | DB 연결 |
| `FileManagementToolkit` | 로컬 파일 읽기·쓰기·목록 | `langchain_community.agent_toolkits` | — |

### 문서검색 (RAG) · 커스텀

| 도구 | 역할 | import |
|---|---|---|
| `create_retriever_tool` | 벡터스토어 검색을 도구로 (RAG) | `langchain_core.tools.retriever` |
| `@tool` / `StructuredTool` | 내 함수·사내 API·DB 조회를 도구로 | `langchain_core.tools` |

> **MCP(Model Context Protocol)**: 최근에는 도구를 LangChain 패키지로 직접 import 하는 대신, 외부 "MCP 서버"(GitHub, 파일시스템, 브라우저 등)를 표준 프로토콜로 연결해 그 서버가 노출한 도구들을 가져다 쓰는 방식도 널리 쓰인다 (`langchain-mcp-adapters`). 어떤 방식이든 결국 **name·description·args 를 가진 호출 가능한 객체** 로 귀결된다.

> 전체 prebuilt 도구 목록은 LangChain 공식 통합(integrations) 문서에서 카테고리별로 확인할 수 있다.

### 4-1. 위키백과 도구 (예시)

`WikipediaQueryRun` 은 위키 문서를 조회하는 도구다. 내부적으로 `WikipediaAPIWrapper` 를 쓴다.

> 실행하려면 `pip`/`uv` 로 `wikipedia` 패키지가 필요하고 네트워크가 있어야 한다. 패키지가 없으면 아래 셀은 안내 메시지를 출력한다.

In [ ]:
try:
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1))
    print("name:", wiki.name)
    print("description:", wiki.description)
    print(wiki.invoke("LangGraph")[:300])
except Exception as e:
    print("위키 도구 실행 불가 (패키지/네트워크 필요):", type(e).__name__, e)
    print("설치: uv add wikipedia")

### 4-2. 웹검색 도구 (예시, 키 필요)

`TavilySearch` 는 실시간 웹검색 도구다. `TAVILY_API_KEY` 가 있어야 실행된다 (`.env` 에 키를 넣고 `load_dotenv()`). 키 없이 구조만 보고 싶으면 아래 셀은 건너뛰어도 된다.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ.get("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch
    web = TavilySearch(max_results=2)
    print("name:", web.name)
    print(web.invoke("What is LangGraph?"))
else:
    print("TAVILY_API_KEY 가 없어 건너뜀 (.env 에 키를 넣으면 실행됨)")

### 4-3. 코드 실행 도구 (예시) — 코드를 실제로 돌려 검증

사용자가 "이 코드 결과 알려줘" 같은 요청을 하면, LLM 이 답을 *추측*하는 대신 **실제로 실행**해 검증할 수 있다. `PythonREPLTool` 이 그 역할을 한다.

> ⚠️ 이 도구는 **임의의 파이썬 코드를 실행**한다. 실제 서비스에서는 반드시 샌드박스/격리 환경에서 써야 한다. 여기서는 학습용으로 안전한 코드만 넣는다.

In [ ]:
from langchain_experimental.tools import PythonREPLTool

py = PythonREPLTool()
print("name:", py.name)
print("desc:", py.description[:70], "...")

# LLM 이 생성했을 법한 코드를 실제로 실행해 결과를 검증
code = """
nums = [n for n in range(1, 21) if n % 3 == 0]
print('3의 배수:', nums)
print('합계:', sum(nums))
"""
print("실행 결과:")
print(py.invoke(code))

## 5. `ToolNode` — 그래프 안에서 도구 실행하기

[basics 복습] 도구를 그래프의 노드로 넣을 때는 LangGraph 의 prebuilt `ToolNode` 를 쓴다. `ToolNode` 는 **마지막 AIMessage 의 `tool_calls` 를 읽어 해당 도구를 실제로 실행**하고, 결과를 `ToolMessage` 로 만들어 messages 에 누적한다.

보통은 LLM 이 `tool_calls` 를 만들어주지만, 여기서는 **LLM 없이** 우리가 호출 의도를 손으로 만들어 ToolNode 가 도구를 실행하는 과정만 따로 확인한다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# [basics 복습] 메시지를 누적하는 State (Annotated + add_messages 리듀서)
class State(TypedDict):
    messages: Annotated[list, add_messages]

# [basics 복습] StateGraph 빌더 → ToolNode 등록 → START/END 연결 → compile
g = StateGraph(State)
g.add_node("tools", ToolNode([multiply, add_tool]))   # 위에서 만든 도구들을 넘김
g.add_edge(START, "tools")
g.add_edge("tools", END)
graph = g.compile()

# LLM 이 만들었을 법한 '도구 호출 의도' 를 직접 만들어 본다
fake_ai = AIMessage(
    content="",
    tool_calls=[
        {"name": "multiply", "args": {"a": 3, "b": 4}, "id": "call_1", "type": "tool_call"},
    ],
)

out = graph.invoke({"messages": [fake_ai]})
for m in out["messages"]:
    print(f"{type(m).__name__:12s} | content={m.content!r} | name={getattr(m, 'name', None)}")

위 결과에서 `ToolMessage` 의 content 가 `'12'` (= 3×4) 인 것을 볼 수 있다. ToolNode 가 호출 의도를 받아 실제로 `multiply` 를 실행한 것이다.

## 정리

- **도구 = name + description + args 를 가진 호출 가능한 객체.** LLM 은 이 셋을 보고 도구를 고른다.
- 만들기: `@tool`(간단) / `@tool(args_schema=...)`(스키마 명시) / `StructuredTool.from_function`(기존 함수)
- prebuilt 도구: 웹검색(`TavilySearch`, `DuckDuckGoSearchRun`), 백과(`WikipediaQueryRun`), 논문(`ArxivQueryRun`), 문서검색(`create_retriever_tool`) 등
- 그래프에선 **`ToolNode`** 가 `tool_calls` 를 읽어 도구를 실행하고 `ToolMessage` 로 결과를 돌려준다

**LLM 에 도구를 실제로 바인딩**(`llm.bind_tools`)해서 LLM 이 스스로 도구를 부르게 하는 전체 흐름은 `projects/02_tool_calling_web_search.ipynb` 에서 다룬다.